# Notebook 03: EfficientNet-B0 Two-Stage ABMIL Training

## 1. Project Setup & Environment Checks
Verify PyTorch CUDA capabilities, dataset path configurations and hardware acceleration availability.

In [1]:
import os
import json
import time
import numpy as np
import gc
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import timm

# Environment check 
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"timm     : {timm.__version__}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

PyTorch  : 2.10.0+cu128
CUDA     : True
timm     : 1.0.26
GPU      : Tesla T4


### 1.1 Input Path & Directory Configuration
Locate input feature arrays, label targets and 5-fold CV splits generated in NB02.

In [2]:
NB02 = Path("/kaggle/input/notebooks/mfjmrizvi/02-mil-patch-extraction")
OUT  = Path("/kaggle/working")

# Input files from NB02
X_TRAIN_PATH        = NB02 / "X_train_patches.npy"
Y_TRAIN_PATH        = NB02 / "y_train_labels.npy"
BAG_IDS_TRAIN_PATH  = NB02 / "bag_ids_train.npy"
CLASS_WEIGHTS_PATH  = NB02 / "class_weights.json"

print("NB02 path exists:", NB02.exists())

NB02 path exists: True


### 1.2 Hyperparameters & Random Seed Initialization
Define global training parameters for Stage 1 patch classification, batch dimensions and model backbones.

In [3]:
MODEL_NAME   = "efficientnet_b0"
PATCH_SIZE   = 224
SEED         = 42

BS_STAGE1    = 128           
EPOCHS_S1    = 15
PATIENCE_S1  = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

torch.manual_seed(SEED)
np.random.seed(SEED)

Device: cuda


### 1.3 Pre-processed Array Verification
Load and check shapes, target label distributions and value ranges for all extracted patch data.

In [4]:
print("Loading arrays...")
X_train_all  = np.load(X_TRAIN_PATH)   
y_train_all  = np.load(Y_TRAIN_PATH)  
bag_ids_all  = np.load(BAG_IDS_TRAIN_PATH)  

fold_ids = np.load(NB02 / "fold_ids.npy")  

with open(CLASS_WEIGHTS_PATH) as f:
    raw_cw = json.load(f)
class_weight_dict = {int(k): float(v) for k, v in raw_cw.items()}

print("\n── Train arrays ──")
print(f"X_train_all : {X_train_all.shape}  dtype={X_train_all.dtype}"
      f"  range=[{X_train_all.min():.3f}, {X_train_all.max():.3f}]")
print(f"y_train_all : {y_train_all.shape}  "
      f"benign={(y_train_all==0).sum()}  "
      f"malignant={(y_train_all==1).sum()}")
print(f"bag_ids_all : {bag_ids_all.shape}  "
      f"unique bags={len(np.unique(bag_ids_all))}")

print(f"\n── Fold structure ──")
print(f"fold_ids : {fold_ids.shape}  unique folds={sorted(set(fold_ids))}")

print("\n── Class weights ──")
print(class_weight_dict)

Loading arrays...

── Train arrays ──
X_train_all : (58820, 224, 224, 1)  dtype=float32  range=[0.000, 1.000]
y_train_all : (58820,)  benign=29225  malignant=29595
bag_ids_all : (58820,)  unique bags=1226

── Fold structure ──
fold_ids : (1226,)  unique folds=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

── Class weights ──
{0: 1.0063301967493585, 1: 0.9937489440783916}


### 1.4 Shared Module Imports
Import core ABMIL components, patch classifiers and utility routines from the central pipeline library.

In [5]:
import sys
sys.path.append('/kaggle/input/datasets/mfjmrizvi/cbis-ddsm-project-config')
from abmil_common import (
    build_backbone, PatchClassifier, get_normalisation_tensors, train_stage1_fold)

## 2. Cross-Validation Holdout Configuration
Isolate validation fold boundaries to prevent data leakage during patch-level backbone feature learning.

In [6]:
fold_ids = np.load(NB02 / "fold_ids.npy")

# Fold 0 used to define the fixed Stage 1 train/val split (backbone stays fixed across folds)
STAGE1_FOLD = 0
all_bags = np.unique(bag_ids_all)
train_bag_indices = all_bags[fold_ids[all_bags] != STAGE1_FOLD]
val_bag_indices   = all_bags[fold_ids[all_bags] == STAGE1_FOLD]

train_mask = np.isin(bag_ids_all, train_bag_indices)
val_mask   = np.isin(bag_ids_all, val_bag_indices)

X_tr, y_tr, bag_tr    = X_train_all[train_mask], y_train_all[train_mask], bag_ids_all[train_mask]
X_val, y_val, bag_val = X_train_all[val_mask],   y_train_all[val_mask],   bag_ids_all[val_mask]

print(f"Stage 1 fixed split (fold {STAGE1_FOLD} held out): "
      f"train_bags={len(train_bag_indices)}  val_bags={len(val_bag_indices)}")

Stage 1 fixed split (fold 0 held out): train_bags=985  val_bags=241


### 2.1 Load Stage 1 Optimal Hyperparameters
Retrieve optimised hyperparameter configurations discovered during Optuna hyperparameter tuning.

In [7]:
#  Load the winning Stage 1 parameters found in NB02.5 
STAGE1_PARAMETER_PATH = Path("/kaggle/input/notebooks/mfjmrizvi/2-6-optuna-run-all/effnet_b0_stage1_optuna_study.json")
with open(STAGE1_PARAMETER_PATH) as f:
    parameter = json.load(f)["best_params"]

OPTIMISER_NAME = parameter["optimiser"]
LR             = parameter["lr"]
WEIGHT_DECAY   = parameter["weight_decay"]
print(f"Using parameter: {parameter}")

all_bags = np.unique(bag_ids_all)

Using parameter: {'optimiser': 'AdamW', 'lr': 1.1527987128232396e-05, 'weight_decay': 0.00757947995334801}


### 2.2 Memory-Efficient Fold Splitter
Construct an index-based data splitter to minimise RAM usage by referencing contiguous array locations during fold splits.

In [8]:
def get_fold_split(fold, all_bags, fold_ids, bag_ids_all, model_name, out_dir):
    if fold == "full":
        rng = np.random.default_rng(999)
        shuffled = rng.permutation(all_bags)
        n_val = int(0.1 * len(shuffled))
        val_bag_indices   = shuffled[:n_val]
        train_bag_indices = shuffled[n_val:]
        save_path = out_dir / f"{model_name}_stage1_full.pth"
    else:
        train_bag_indices = all_bags[fold_ids[all_bags] != fold]
        val_bag_indices   = all_bags[fold_ids[all_bags] == fold]
        save_path = out_dir / f"{model_name}_stage1_fold{fold}.pth"

    train_idx = np.where(np.isin(bag_ids_all, train_bag_indices))[0]
    val_idx   = np.where(np.isin(bag_ids_all, val_bag_indices))[0]

    return train_idx, val_idx, save_path

## 3. Stage 1 Cross-Validation Training Loop
Execute Stage 1 patch-level training across all 5 cross-validation folds plus the full production set.

In [9]:
fold_histories = {}

for fold in [0, 1, 2, 3, 4, "full"]:
    print(f"\n{'='*60}\nTraining Stage 1: {MODEL_NAME} fold={fold}\n{'='*60}")

    train_idx, val_idx, save_path = get_fold_split(
        fold, all_bags, fold_ids, bag_ids_all, MODEL_NAME, OUT
    )
    print(f"Train patches: {len(train_idx)}  Val patches: {len(val_idx)}")

    history, best_val_loss = train_stage1_fold(
        model_name=MODEL_NAME,
        X_all=X_train_all, y_all=y_train_all,
        train_idx=train_idx, val_idx=val_idx,
        class_weight_dict=class_weight_dict,
        optimiser_name=OPTIMISER_NAME, lr=LR, weight_decay=WEIGHT_DECAY,
        save_path=save_path, device=DEVICE,
        epochs=EPOCHS_S1, patience=PATIENCE_S1
    )

    fold_histories[str(fold)] = {
        "best_val_loss": best_val_loss,
        "parameters": parameter,
        "epochs_budget": EPOCHS_S1,
        "patience": PATIENCE_S1,
        "train_patches": len(train_idx),
        "val_patches": len(val_idx),
        "history": history,
    }

    gc.collect(); torch.cuda.empty_cache()


Training Stage 1: efficientnet_b0 fold=0
Train patches: 47252  Val patches: 11568


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

  Ep 01/15  train=0.6516  val=0.6572  auc=0.6473
    ✓ Saved: efficientnet_b0_stage1_fold0.pth
  Ep 02/15  train=0.6028  val=0.6165  auc=0.6955
    ✓ Saved: efficientnet_b0_stage1_fold0.pth
  Ep 03/15  train=0.5760  val=0.6119  auc=0.6973
    ✓ Saved: efficientnet_b0_stage1_fold0.pth
  Ep 04/15  train=0.5534  val=0.6236  auc=0.6932
  Ep 05/15  train=0.5314  val=0.6220  auc=0.6966
  Ep 06/15  train=0.5092  val=0.6318  auc=0.6935
  Ep 07/15  train=0.4841  val=0.6373  auc=0.7018
  Ep 08/15  train=0.4572  val=0.6802  auc=0.6785
  Early stopping at epoch 8

Training Stage 1: efficientnet_b0 fold=1
Train patches: 47355  Val patches: 11465
  Ep 01/15  train=0.6516  val=0.6650  auc=0.5978
    ✓ Saved: efficientnet_b0_stage1_fold1.pth
  Ep 02/15  train=0.5930  val=0.6619  auc=0.6163
    ✓ Saved: efficientnet_b0_stage1_fold1.pth
  Ep 03/15  train=0.5628  val=0.6683  auc=0.6229
  Ep 04/15  train=0.5386  val=0.6840  auc=0.6188
  Ep 05/15  train=0.5177  val=0.6896  auc=0.6176
  Ep 06/15  train=0.49

## 4. Stage 1 Summary & Model Checkpoint Persistence
Save fold histories, loss trajectories and training evaluation summaries to disk.

In [10]:
with open(OUT / f"{MODEL_NAME}_stage1_all_folds_summary.json", "w") as f:
    json.dump(fold_histories, f, indent=2)

print("\nAll 6 Stage 1 backbones trained (5 CV folds + 1 production).")


All 6 Stage 1 backbones trained (5 CV folds + 1 production).
